# Correcting the cell masks

The three annotated regions were segmented differently: the lung region holds cell
nuclei, while EMMA1 and X02 hold whole cells. This notebook brings them onto the same
footing by growing the nuclei and shrinking the whole cells, so that every region ends
up with a nucleus-sized object per cell.

Only the geometry changes. Each cell keeps its own positivity value (1 negative,
2 positive, 3 unknown) in every channel, and the geometry is the same for all channels,
exactly as in the originals.

A last cell turns the separately numbered nuclei masks into binary images, pulling
apart the nuclei that share a border.

Everything here keeps the cells separated by background, which is what lets a connected
component analysis pick them apart, so `TrainModel.ipynb` keeps working.

In [ ]:
import numpy as np
import nibabel as nib
import tifffile
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.segmentation import find_boundaries, watershed

DATA_DIR = "TrainData"

EXPANSION_PX = 6    # how far to grow the lung nuclei
NUCLEUS_PX = 50     # an 8 um nucleus at 1 um/pixel covers about 50 pixels

NEGATIVE, POSITIVE, UNKNOWN = 1, 2, 3    # the values used in the positivity masks


def load_labels(filename):
    """Return the volume, one label per cell, and each cell's value per channel."""
    image = nib.load(f"{DATA_DIR}/{filename}")
    data = np.asanyarray(image.dataobj)

    cells, n_cells = ndimage.label(data[:, :, 0] > 0)

    # The cells are uniform within a channel, so one pixel each gives every value.
    representative = np.zeros(n_cells + 1, dtype=np.int64)
    representative[cells.ravel()] = np.arange(cells.size)
    values = data.reshape(-1, data.shape[2])[representative[1:]]

    return image, cells, values


def save_labels(image, cells, values, filename):
    """Paint the per channel values back onto the new geometry and write it out."""
    table = np.vstack([np.zeros((1, values.shape[1]), values.dtype), values])
    data = table[cells]

    out = nib.Nifti1Image(data, image.affine, image.header)
    out.set_data_dtype(data.dtype)
    nib.save(out, f"{DATA_DIR}/{filename}")

    kept = ndimage.label(data[:, :, 0] > 0)[1]
    sizes = np.bincount(cells.ravel())[1:]
    print(f"  wrote {filename}")
    print(f"  {kept} separate cells, median {np.median(sizes):.0f} px "
          f"(min {sizes.min()}, max {sizes.max()})")
    return data


def separate_cells(cells):
    """Clear the pixels of the lower numbered cell wherever two cells touch.

    Clearing one side of a contact is enough. A pixel only survives when none of its
    neighbours belongs to a higher numbered cell, so nothing that survives can still be
    beside another cell, and the two end up one pixel apart.
    """
    highest = ndimage.maximum_filter(cells, size=3)

    return np.where((cells > 0) & (highest > cells), 0, cells)


def keep_largest_piece(cells):
    """Reduce every cell to its largest connected piece.

    Growing and shrinking can both break a cell into several pieces, and a connected
    component analysis would then count each piece as a separate cell.
    """
    pieces, n_pieces = ndimage.label(cells > 0)
    sizes = np.bincount(pieces.ravel(), minlength=n_pieces + 1)

    # Pieces never straddle two cells, so each one has a single owner.
    owner = np.zeros(n_pieces + 1, dtype=cells.dtype)
    owner[pieces.ravel()] = cells.ravel()

    largest = np.zeros(int(cells.max()) + 1, dtype=np.int64)
    for piece in np.argsort(sizes[1:]) + 1:   # smallest first, so the largest wins
        largest[owner[piece]] = piece

    return np.where(pieces == largest[cells], cells, 0)


def expand_cells(cells, distance):
    """Grow every cell by `distance` pixels, flooding outwards from the cells.

    This is the expansion Opal Studio uses: a watershed over the distance to the
    nearest cell, with a watershed line left standing so neighbours never merge.
    """
    outside = ndimage.distance_transform_edt(cells == 0)

    markers = np.where(find_boundaries(cells, mode="inner"), 0, cells)

    # A cell under two pixels across is all boundary and would lose its marker
    # completely, so those get their centre put back.
    missing = np.setdiff1d(np.unique(cells), np.unique(markers))
    missing = missing[missing > 0]
    for cell_id, centre in zip(missing, ndimage.center_of_mass(cells > 0, cells, missing)):
        markers[int(centre[0]), int(centre[1])] = cell_id

    grown = watershed(outside.astype(np.float32), markers=markers.astype(np.int32),
                      mask=outside <= distance, watershed_line=True)

    # The watershed line only separates neighbours left and right and up and down, and
    # can leave two cells meeting at a corner, so they are pulled apart properly here.
    return keep_largest_piece(separate_cells(grown))


def shrink_cells(cells, target):
    """Erode every cell down to about `target` pixels, keeping its most central ones.

    Cells that are already at or below the target are left untouched.
    """
    depth = ndimage.distance_transform_edt(cells > 0)

    labels = cells.ravel()
    inside = np.flatnonzero(labels)

    # Group the pixels by cell, deepest first, and keep the first `target` of each.
    inside = inside[np.lexsort((-depth.ravel()[inside], labels[inside]))]
    grouped = labels[inside]
    rank = np.arange(inside.size) - np.searchsorted(grouped, grouped, side="left")

    shrunk = np.zeros_like(cells)
    keep = inside[rank < target]
    shrunk.ravel()[keep] = labels[keep]

    return keep_largest_piece(shrunk)


# ---------------------------------------------------------------------------
# 1. Lung: grow the nuclei into whole cells.
# ---------------------------------------------------------------------------
name = "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_positivity_corrected"

image, cells, values = load_labels(f"{name}.nii.gz")
before = np.bincount(cells.ravel())[1:]

grown = expand_cells(cells, EXPANSION_PX)

apart = ndimage.label(grown > 0)[1]
apart_diagonal = ndimage.label(grown > 0, structure=np.ones((3, 3)))[1]
assert apart == apart_diagonal == len(before),     f"{len(before)} cells became {apart} objects ({apart_diagonal} counting corners)"

print(f"{name[:38]}: {len(before)} nuclei, median {np.median(before):.0f} px "
      f"-> grown by {EXPANSION_PX} px")
save_labels(image, grown, values, f"{name}_dilated.nii.gz")

## EMMA1: shrink the whole cells to nuclei

In [ ]:
image, cells, values = load_labels("EMMA1_inference_corrected.nii")
before = np.bincount(cells.ravel())[1:]

shrunk = shrink_cells(cells, NUCLEUS_PX)

print(f"EMMA1: {len(before)} cells, median {np.median(before):.0f} px, "
      f"{(before <= NUCLEUS_PX).sum()} already <= {NUCLEUS_PX} px and left alone")
save_labels(image, shrunk, values, "EMMA1_inference_corrected_eroded.nii")

## X02: shrink the whole cells to nuclei

X02's cell mask lives in `X02_a0_full_mask_modified.nii.gz` — the file paired with
`X02_a0_full.nii.gz` in `TrainModel.ipynb`. `X02_a0_full.nii.gz` itself is the marker
stack and holds no cells.

In [ ]:
image, cells, values = load_labels("X02_a0_full_mask_modified.nii.gz")
before = np.bincount(cells.ravel())[1:]

shrunk = shrink_cells(cells, NUCLEUS_PX)

print(f"X02: {len(before)} cells, median {np.median(before):.0f} px, "
      f"{(before <= NUCLEUS_PX).sum()} already <= {NUCLEUS_PX} px and left alone")
save_labels(image, shrunk, values, "X02_a0_full_eroded.nii.gz")

## Nuclei masks: make them binary and pull the touching nuclei apart

These masks number every nucleus separately and let neighbours share a border. The rest
of the pipeline recovers cells with a connected component analysis, so they have to
become plain binary images with background in between.

Only one side of each contact is cleared, which leaves a single pixel of background
between two nuclei and costs half as many pixels as clearing both sides.

In [ ]:
NUCLEI_MASKS = [
    "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask.ome.tiff",
    "EMMA1_cell_nuclei_mask.ome.tiff",
]


for filename in NUCLEI_MASKS:
    with tifffile.TiffFile(f"{DATA_DIR}/{filename}") as tif:
        nuclei = tif.series[0].asarray().astype(np.int32)

    n_nuclei = int((np.bincount(nuclei.ravel())[1:] > 0).sum())
    n_pixels = int((nuclei > 0).sum())

    binary = (keep_largest_piece(separate_cells(nuclei)) > 0).astype(np.uint8)

    # Every nucleus must survive as exactly one object, whether or not corners count.
    apart = ndimage.label(binary)[1]
    apart_diagonal = ndimage.label(binary, structure=np.ones((3, 3)))[1]
    assert apart == apart_diagonal == n_nuclei, \
        f"{n_nuclei} nuclei became {apart} objects ({apart_diagonal} counting corners)"

    out_name = filename.replace(".ome.tiff", "_corrected.ome.tiff")
    tifffile.imwrite(f"{DATA_DIR}/{out_name}", binary, photometric="minisblack", ome=True)

    print(f"{filename[:52]}")
    print(f"  {n_nuclei} nuclei, {n_pixels - int(binary.sum())} of {n_pixels} pixels "
          f"cleared ({1 - binary.sum() / n_pixels:.1%}) to pull them apart")
    print(f"  wrote {out_name}")

## Give the nuclei a positivity value for every channel

The separated nuclei carry no positivity yet. Each one takes the value of the cell it
sits in, read from a volume that already has one, at the nucleus centre.

The nuclei masks come from OME-TIFF and the volumes from NIfTI, and the two do not
always agree on how the image is laid out: EMMA1 stores its OME-TIFF transposed with
respect to its NIfTI, while the lung region stores both the same way. The volumes are
also read the way `TrainModel.ipynb` reads them, since the masks are stored back to
front relative to the marker stacks and only line up once reoriented. Both orientations
were confirmed against the DNA channels, where nuclei have to be bright.

In [ ]:
NUCLEI_TO_LABEL = [
    # nuclei mask, volume to read the values from, how the mask sits in it, output
    ("20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected.ome.tiff",
     "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_positivity_corrected_dilated.nii.gz",
     "identity",
     "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled.nii"),
    ("EMMA1_cell_nuclei_mask_corrected.ome.tiff",
     "EMMA1_inference_corrected.nii",
     "transpose",
     "EMMA1_inference_cell_nuclei_mask_corrected_labled.nii"),
]

ORIENTATION = {"identity": lambda a: a, "transpose": lambda a: a.T}


for nuclei_file, volume_file, orientation, out_name in NUCLEI_TO_LABEL:
    nuclei = ORIENTATION[orientation](tifffile.imread(f"{DATA_DIR}/{nuclei_file}"))

    image = nib.as_closest_canonical(nib.load(f"{DATA_DIR}/{volume_file}"))
    volume = np.asanyarray(image.dataobj)

    if nuclei.shape != volume.shape[:2]:
        raise ValueError(f"nuclei are {nuclei.shape}, volume is {volume.shape[:2]}")

    cells, n_nuclei = ndimage.label(nuclei > 0)
    centres = np.array(ndimage.center_of_mass(
        cells > 0, cells, np.arange(1, n_nuclei + 1))).astype(int)

    values = volume[centres[:, 0], centres[:, 1], :]

    # A centre can fall just outside the cell the nucleus belongs to. Those nuclei take
    # the commonest value under their whole footprint instead, and count as unknown
    # when there is no cell under them at all.
    stray = np.flatnonzero(values[:, 0] == 0)
    empty = 0

    for i in stray:
        under = volume[cells == i + 1]
        if not (under > 0).any():
            empty += 1
        for z in range(volume.shape[2]):
            seen = under[:, z][under[:, z] > 0]
            values[i, z] = np.bincount(seen).argmax() if seen.size else UNKNOWN

    print(f"{nuclei_file[:52]}")
    print(f"  {n_nuclei} nuclei x {volume.shape[2]} channels, "
          f"{len(stray)} centres landed outside a cell ({empty} with no cell under them)")
    print(f"  negative {int((values == NEGATIVE).sum())}, "
          f"positive {int((values == POSITIVE).sum())}, "
          f"unknown {int((values == UNKNOWN).sum())}")
    save_labels(image, cells, values, out_name)

## Grow the labelled nuclei out to a range of radii

The same expansion Opal Studio uses: a watershed flooding outwards from the nuclei over
the distance to the nearest one, with a watershed line left standing between neighbours
so that two of them never merge.

The watershed line only separates neighbours left and right and up and down, and can
leave two cells meeting at a corner, so the cells are pulled apart afterwards in the same
way as everywhere else in this notebook. That keeps exactly one pixel between them.

In [ ]:
DILATE_MASKS = [
    "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled.nii",
    "EMMA1_inference_cell_nuclei_mask_corrected_labled.nii",
]

DILATIONS = range(1, 11)


for filename in DILATE_MASKS:
    image, cells, values = load_labels(filename)
    n_cells = int(cells.max())

    print(f"{filename[:58]}  {n_cells} cells")

    for distance in DILATIONS:
        grown = expand_cells(cells, distance)

        # every cell must still be there, as exactly one object, with a gap around it
        apart = ndimage.label(grown > 0)[1]
        apart_diagonal = ndimage.label(grown > 0, structure=np.ones((3, 3)))[1]
        assert apart == apart_diagonal == n_cells,             f"{n_cells} cells became {apart} objects at {distance} px"

        save_labels(image, grown, values, filename.replace(".nii", f"_dilated{distance}.nii"))